In [103]:
### parallel workflow, lets try something simple
from langgraph.graph   import START , END, StateGraph
from langchain_ollama import OllamaLLM
from langchain_openai   import ChatOpenAI
from pydantic import BaseModel, Field
import operator
from typing import TypedDict, Annotated
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
# from langchain.output_parsers.fix import OutputFixingParser
from dotenv import load_dotenv

START---1_branch--essay clarity and score
     ---2_branch--essay depth and score
     ---3_branch--essay language and score

In [104]:
load_dotenv()

True

In [105]:
model = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
# model = OllamaLLM(model="qwen2.5:3b", temperature=0.2)
# model = OllamaLLM(model="gemma4:e2b", temperature=0.2)

In [106]:
##pydantic schemas    
# 
class LanguageSchema(BaseModel):
    language_feedback: str = Field(..., description="The feedback on the language of the essay")
    language_score: int = Field(..., description="The score on the langauge of the essay, range from 0 to 10", le=10, ge=0)

class ClaritySchema(BaseModel):
    clarity_feedback: str = Field(..., description="The feedback on the clarity of the essay")
    clarity_score: int = Field(..., description="The score on the clarity of the essay, range from 0 to 10", le=10, ge=0)

class CreativitySchema(BaseModel):
    creativity_feedback: str = Field(..., description="The feedback on the creativity of the essay")
    creativity_score: int = Field(..., description="The score on the creativity of the essay, range from 0 to 10", le=10, ge=0)

class AnalysisSchema(BaseModel):
    analysis_feedback: str = Field(..., description="The feedback on the analysis of the essay")
    analysis_score: int = Field(..., description="The score on the analysis of the essay, range from 0 to 10", le=10, ge=0)
    
class OverallSchema(BaseModel):
    overall_feedback: str = Field(..., description="The overall feedback of the essay")
    

In [107]:
class EssayState(TypedDict):
    essay: str

    language_feedback: str
    clarity_feedback: str
    creativity_feedback: str
    analysis_feedback: str
    overall_feedback: str

    language_score: int
    clarity_score: int
    creativity_score: int
    analysis_score: int

    total_score: Annotated[int, operator.add]
    average_score: float

In [108]:
## parsers

language_parser = PydanticOutputParser(pydantic_object=LanguageSchema)
clarity_parser = PydanticOutputParser(pydantic_object=ClaritySchema)
creativity_parser = PydanticOutputParser(pydantic_object=CreativitySchema)
analysis_parser = PydanticOutputParser(pydantic_object=AnalysisSchema)

overall_parser = PydanticOutputParser(pydantic_object=OverallSchema)


In [109]:
##prompt template

language_prompt = PromptTemplate(template=""" you are an expert academic essay evaluator assessing the language quality only.

Evaluate the following essay and provide feedback on langauge and a score from 0 to 10, where 10 is the best.
Essay:
{essay}

Return ONLY a valid JSON object with no extra text, following this format:
{{ 
"language_feedback": "your feedback on the language of the essay",
"language_score": your score on the language of the essay  
}}
                                """, input_variables=["essay"])


clarity_prompt = PromptTemplate(template=""" you are an expert academic essay evaluator assessing the clarity only.
Evaluate the following essay and provide feedback on clarity and a score from 0 to 10, where 10 is the best.
Essay:
{essay}
Return ONLY a valid JSON object with no extra text, following this format:
{{  
"clarity_feedback": "your feedback on the clarity of the essay",
"clarity_score": your score on the clarity of the essay
}}
                                """, input_variables=["essay"])

creativity_prompt = PromptTemplate(template=""" you are an expert academic essay evaluator assessing the creativity only.
Evaluate the following essay and provide feedback on creativity and a score from 0 to 10, where 10 is the best.
Essay:
{essay} 
Return ONLY a valid JSON object with no extra text, following this format:
{{
"creativity_feedback": "your feedback on the creativity of the essay",
"creativity_score": your score on the creativity of the essay
}}
                                """, input_variables=["essay"])

analysis_prompt = PromptTemplate(template=""" you are an expert academic essay evaluator assessing the analysis only.
Evaluate the following essay and provide feedback on analysis and a score from 0 to 10, where 10 is the best.
Essay:
{essay}
Return ONLY a valid JSON object with no extra text, following this format:
{{
"analysis_feedback": "your feedback on the analysis of the essay",
"analysis_score": your score on the analysis of the essay
}}

                                """, input_variables=["essay"])


overall_prompt = PromptTemplate(
    template="""You are a senior academic essay evaluator.

Essay:
{essay}

Individual Feedback:
- Language: {language_feedback}
- Clarity: {clarity_feedback}
- Creativity: {creativity_feedback}
- Analysis: {analysis_feedback}

Average Score: {average_score}

Write a cohesive 4-5 sentence paragraph highlighting strengths and weaknesses.

Return ONLY valid JSON:
{{
  "overall_feedback": "<paragraph>"
}}
""",
    input_variables=[
        "essay",
        "language_feedback",
        "clarity_feedback",
        "creativity_feedback",
        "analysis_feedback",
        "average_score",
    ],
)

In [110]:
## define the evaluation schema for each aspect
def check_language(state: EssayState):
    response = (language_prompt | model).invoke({"essay": state["essay"]})
    raw = response.content if hasattr(response, "content") else str(response)

    r = language_parser.parse(raw).model_dump()
    score = r["language_score"]

    print(f"[language] score {score}/10")

    return {
        "language_feedback": r["language_feedback"],
        "language_score": score,
        "total_score": score,  # reducer adds
    }

def check_clarity(state: EssayState):
    response = (clarity_prompt | model).invoke({"essay": state["essay"]})
    raw = response.content if hasattr(response, "content") else str(response)

    r = clarity_parser.parse(raw).model_dump()
    score = r["clarity_score"]

    print(f"[clarity] score {score}/10")

    return {
        "clarity_feedback": r["clarity_feedback"],
        "clarity_score": score,
        "total_score": score,
    }
    
def check_creativity(state: EssayState):
    response = (creativity_prompt | model).invoke({"essay": state["essay"]})
    raw = response.content if hasattr(response, "content") else str(response)

    r = creativity_parser.parse(raw).model_dump()
    score = r["creativity_score"]

    print(f"[creativity] score {score}/10")

    return {
        "creativity_feedback": r["creativity_feedback"],
        "creativity_score": score,
        "total_score": score,
    }
    
def check_analysis(state: EssayState):
    response = (analysis_prompt | model).invoke({"essay": state["essay"]})
    raw = response.content if hasattr(response, "content") else str(response)

    r = analysis_parser.parse(raw).model_dump()
    score = r["analysis_score"]

    print(f"[analysis] score {score}/10")

    return {
        "analysis_feedback": r["analysis_feedback"],
        "analysis_score": score,
        "total_score": score,
    }

def overall_evaluation(state: EssayState):
    total = state["total_score"]
    avg = total / 4.0

    print(f"[overall] average score {avg}/10 (total={total})")

    response = (overall_prompt | model).invoke({
        "essay": state["essay"],
        "language_feedback": state["language_feedback"],
        "clarity_feedback": state["clarity_feedback"],
        "creativity_feedback": state["creativity_feedback"],
        "analysis_feedback": state["analysis_feedback"],
        "average_score": round(avg, 2),
    })

    raw = response.content if hasattr(response, "content") else str(response)
    parsed = overall_parser.parse(raw)

    return {
        "overall_feedback": parsed.overall_feedback,
        "average_score": round(avg, 2),
    }

In [111]:
graph = StateGraph(EssayState)

graph.add_node("check_language",check_language)
graph.add_node("check_clarity",check_clarity)
graph.add_node("check_creativity",check_creativity)
graph.add_node("check_analysis",check_analysis)
graph.add_node("overall_evaluation",overall_evaluation)

graph.add_edge(START, "check_language")
graph.add_edge(START, "check_clarity")
graph.add_edge(START, "check_creativity")
graph.add_edge(START, "check_analysis")

graph.add_edge("check_language", "overall_evaluation")
graph.add_edge("check_clarity", "overall_evaluation")
graph.add_edge("check_creativity", "overall_evaluation")
graph.add_edge("check_analysis", "overall_evaluation")
graph.add_edge("overall_evaluation", END)

workflow = graph.compile()



In [112]:
def evaluate_essay(essay: str):
    initial_state = {
        "essay": essay,

        "language_feedback": "",
        "clarity_feedback": "",
        "creativity_feedback": "",
        "analysis_feedback": "",
        "overall_feedback": "",

        "language_score": 0,
        "clarity_score": 0,
        "creativity_score": 0,
        "analysis_score": 0,

        "total_score": 0,
        "average_score": 0.0,
    }

    return workflow.invoke(initial_state)

In [113]:
def print_report(state: EssayState):
    print("\n=== Essay Evaluation Report ===")
    print(f"Essay: {state['essay']}\n")
    print(f"Language Feedback: {state['language_feedback']}")
    print(f"Clarity Feedback: {state['clarity_feedback']}")
    print(f"Creativity Feedback: {state['creativity_feedback']}")
    print(f"Analysis Feedback: {state['analysis_feedback']}")
    print(f"Overall Feedback: {state['overall_feedback']}")
    print(f"Total Score: {state['total_score']}/40")
    print(f"Average Score: {state['average_score']}/10")

In [114]:
Sample_essay= """

In Gotham City there is a man called Batman who fight crime but not like police. He is rich person also and name is bruce wayn but people dont know that much. Gotham is very dark and danger place and many bad peoples live there and doing crimes everyday and night also. Batman go outside in night and beat criminals and make them scared so they not do bad things again but sometimes they still do.

Batman dont have any superpowers like superman but he is still very strong because he train alot and have many gadgets like batarang and car which is called batmobile and also many suits. He use brain also because he is very inteligent but sometimes he also just fight without thinking properly and get hurt. Police sometimes help him but also sometimes they dont like him because he is not legal person.

There is also joker who is very crazy villan and he do many bad jokes and crimes and batman always trying to stop him but joker always coming back which is annoying and confusing also. Gotham city is not becoming better even after batman doing so much work which makes it little useless maybe but still he try.

Bruce wayn in day time is acting like normal rich man and doing bussiness and helping city but in night he become batman which is secret. This is double life and it is difficult but he still doing it. Sometimes it feels like he is too serious and dont enjoy life much.

In the end batman is good hero but also little strange because he scare people and fight alot. Gotham is still bad place and maybe will always be like that. Batman try but not fully success.


"""

In [115]:
final_state = evaluate_essay(Sample_essay)

[creativity] score 4/10
[language] score 3/10
[clarity] score 3/10
[analysis] score 4/10
[overall] average score 3.5/10 (total=14)


In [116]:
print_report(final_state)


=== Essay Evaluation Report ===
Essay: 

In Gotham City there is a man called Batman who fight crime but not like police. He is rich person also and name is bruce wayn but people dont know that much. Gotham is very dark and danger place and many bad peoples live there and doing crimes everyday and night also. Batman go outside in night and beat criminals and make them scared so they not do bad things again but sometimes they still do.

Batman dont have any superpowers like superman but he is still very strong because he train alot and have many gadgets like batarang and car which is called batmobile and also many suits. He use brain also because he is very inteligent but sometimes he also just fight without thinking properly and get hurt. Police sometimes help him but also sometimes they dont like him because he is not legal person.

There is also joker who is very crazy villan and he do many bad jokes and crimes and batman always trying to stop him but joker always coming back which 